# Como Carregar tabelas?
[Documentação Input/output](https://pandas.pydata.org/docs/reference/io.html)

Podemos carregar (e salvar) vários tipos de arquivos utilizando o pandas, arquivos `CSV` e `parquet` são os mais utilizados, no entanto o pandas nos permite lidar bem com arquivos:
- **pickle**
- **csv**
- **excel**
- **clipboard**
- **json**
- **html**
- **xml**
- **latex**
- **hdf**
- **parquet**
- **sas**
- **sql**
- **Google BigQuery (gbq)**

Para leitura na maioria dos casos vamos utilizar o *prefixo* `read_<tipo do arquivo (csv, parquet, etc...)>` seguido pelo tipo do arquivo:
```python
import pandas as pd

df = pd.read_csv('caminho_do_meu_arquivo_csv.csv')  # faço o load do arquivo
df = df.dropna()  # faço algo com meu dataframe
df.to_csv('caminho_onde_quero_salvar_meu_dataframe.csv')  # salvo o que eu fiz
```
Isso pode ser feito com qualquer uma das extensões de arquivo listadas acima.

Lembre-se dado que conseguimos salvar o dataframe também conseguimos fazer o load (ou `read`) para carregar ele em memória.

In [1]:
import sqlite3
import pandas as pd

from pathlib import Path

In [2]:
PATH_DATABASE = Path('../../database/database.db')  # onde meu database está
conn = sqlite3.connect(PATH_DATABASE)  # conexão com o database

In [3]:
query = '''
SELECT
    *
FROM
    customer_churn_records
'''
df = pd.read_sql(query, conn)  # lê a tabela
conn.close()  # fecha a conexão

In [4]:
df.head()

,customer_id,surname,credit_score,geography,gender,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,complain,satisfaction_score,card_type,point_earned
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


## Salvando arquivos

### Pickle

In [10]:
PATH_TO_SAVE = Path('utils/03-HowLoadandSaveDatasets')
df.to_pickle(PATH_TO_SAVE / 'dataset_pickle.pkl')

### CSV

In [20]:
PATH_TO_SAVE = Path('utils/03-HowLoadandSaveDatasets')
df.to_csv(
    PATH_TO_SAVE / 'dataset_csv.csv',
    index=False,  # remove a coluna de index
    sep=';',  # tipo de separação
    decimal='.',  # como lidar com decimais (floats)
    header=True  # adiconar header
)

### Excel

In [12]:
PATH_TO_SAVE = Path('utils/03-HowLoadandSaveDatasets')
SHEET_NAME = 'Dataset'
df.to_excel(PATH_TO_SAVE / 'dataset_excel.xlsx', sheet_name=SHEET_NAME)

### JSON

Com json podemos configurar a sua **orientação** ela pode ser:
- split
- records;
- index;
- columns;
- values;
- table;

`SPLIT`: Ele É O MAIS "fiel" a um DataFrame. A estrutura é dividida em 3 partes, **index, columns, data**:
```json
{
  "index": [0,1],
  "columns": ["a","b"],
  "data": [[1,2], [3,4]]
}
```

`RECORDS`: Cada linha vira um dicionário.
```json
[
  {"a":1, "b":2},
  {"a":3, "b":4}
]
```

`INDEX`: As chaves do JSON são os indices, valores são dicionários onde as colunas são as chaves.
```json
{
  "0": {"a":1, "b":2},
  "1": {"a":3, "b":4}
}
```

`COLUMNS`: As colunas são as chaves que mapeiam para os valores.
```json
{
  "a": {"0":1, "1":3},
  "b": {"0":2, "1":4}
}
```

`VALUES`: Só os valores, não existe index nem colunas. Ideal para transformar em tensores numpy ou pytorch.
```json
[
  [1,2],
  [3,4]
]
```

`TABLE`: Formato mais robusto, próprio para armazenamento orientado a schema.
```json
{
  "schema": {...},
  "data": [
    {"a":1, "b":2},
    {"a":3, "b":4}
  ]
}
```

✅ Resumo
| orient      | Estrutura              | Quando usar                          |
| ----------- | ---------------------- | ------------------------------------ |
| **split**   | index, columns, data   | Para reconstruir o DF fielmente      |
| **records** | lista de dicts         | APIs, JSON comum, frontend           |
| **index**   | dict index -> linha    | Quando index é chave primária        |
| **columns** | dict coluna -> valores | Pipelines orientados a colunas       |
| **values**  | pura matriz            | ML, NumPy, PyTorch                   |
| **table**   | schema + data          | Armazenamento confiável, longo prazo |


In [27]:
print(df.iloc[:2, :2].to_json(orient='records'))

[{"customer_id":15634602,"surname":"Hargrave"},{"customer_id":15647311,"surname":"Hill"}]


### HTML

In [24]:
print(df.iloc[:2, :2].to_html())

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>customer_id</th>
      <th>surname</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>15634602</td>
      <td>Hargrave</td>
    </tr>
    <tr>
      <th>1</th>
      <td>15647311</td>
      <td>Hill</td>
    </tr>
  </tbody>
</table>


### XML

In [29]:
print(df.iloc[:2, :2].to_xml())

<?xml version='1.0' encoding='utf-8'?>
<data>
  <row>
    <index>0</index>
    <customer_id>15634602</customer_id>
    <surname>Hargrave</surname>
  </row>
  <row>
    <index>1</index>
    <customer_id>15647311</customer_id>
    <surname>Hill</surname>
  </row>
</data>


### LaTex

In [32]:
print(df.iloc[:2, :2].to_latex())

\begin{tabular}{lrl}
\toprule
 & customer_id & surname \\
\midrule
0 & 15634602 & Hargrave \\
1 & 15647311 & Hill \\
\bottomrule
\end{tabular}



### Parquet

In [5]:
PATH_TO_SAVE = Path('utils/03-HowLoadandSaveDatasets')
df.to_parquet(PATH_TO_SAVE / 'dataset_parquet.parquet')